In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import requests
import json
import time
import pandas as pd
from datetime import datetime, timezone, timedelta
import os

# --- 1. Session加载函数 ---
def load_session_with_cookies(uid, cookies_path="weibo_cookies.json"):
    """
    加载Cookies并根据提供的真实Request Headers构建一个完整的、高仿真的请求头。
    uid: 用户的ID，用于动态生成正确的Referer。
    """
    try:
        with open(cookies_path, "r") as f:
            cookies_list = json.load(f)

        session = requests.Session()
        for cookie in cookies_list:
            session.cookies.set(cookie["name"], cookie["value"])

        xsrf_token = ""
        for cookie in cookies_list:
            if cookie["name"] == "XSRF-TOKEN":
                xsrf_token = cookie["value"]
                break

        headers = {
            "accept": "application/json, text/plain, */*",
            "accept-language": "zh-CN,zh;q=0.9",
            "client-version": "v2.47.120",
            "priority": "u=1, i",
            "referer": f"https://weibo.com/u/{uid}",
            "sec-ch-ua": '"Google Chrome";v="141", "Not?A_Brand";v="8", "Chromium";v="141"',
            "sec-ch-ua-mobile": "?0",
            "sec-ch-ua-platform": '"Windows"',
            "sec-fetch-dest": "empty",
            "sec-fetch-mode": "cors",
            "sec-fetch-site": "same-origin",
            "server-version": "v2025.09.29.1",
            "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36",
            "x-requested-with": "XMLHttpRequest",
            "x-xsrf-token": xsrf_token,
        }

        session.headers.update(headers)

        if not xsrf_token:
            print("警告: 未在Cookies中找到 XSRF-TOKEN，这可能导致请求失败。")

        print("Session、Cookies和高仿真Headers加载成功。")
        return session

    except FileNotFoundError:
        print(f"错误: {cookies_path} 文件未找到。请先运行登录代码获取Cookies。")
        return None
    except Exception as e:
        print(f"加载Session时发生未知错误: {e}")
        return None

# --- 2. 微博爬取主函数 (已移除评论) ---
def scrape_official_account(uid, start_date_str, end_date_str=None, originals_only=False):
    """
    爬取指定UID用户在指定日期区间内的微博。
    (已修改：保存文件时会包含UID，并自动创建目录)
    """
    session = load_session_with_cookies(uid=uid) 
    if not session:
        return

    start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
    end_date = datetime.strptime(end_date_str, '%Y-%m-%d') if end_date_str else datetime.now()
    end_date = end_date.replace(hour=23, minute=59, second=59)

    print(f"设定爬取区间: 从 {start_date.strftime('%Y-%m-%d')} 到 {end_date.strftime('%Y-%m-%d')}")
    print(f"微博筛选模式: {'仅保留原创微博' if originals_only else '包含原创和转发微博'}")

    posts_api = "https://weibo.com/ajax/statuses/mymblog"
    page = 1
    all_posts_data = []
    
    stop_scraping = False

    print(f"开始爬取用户 {uid} 的微博...")
    while not stop_scraping:
        params = {'uid': uid, 'page': page, 'feature': 0}
        try:
            response = session.get(posts_api, params=params)
            response.raise_for_status()
            data = response.json()

            posts = data.get('data', {}).get('list', [])
            if not posts:
                print("已获取所有微博页面，任务结束。")
                break

            print(f"正在处理第 {page} 页的微博...")

            for post in posts:
                created_at_str = post.get('created_at')
                try:
                    post_date_aware = datetime.strptime(created_at_str, '%a %b %d %H:%M:%S %z %Y')
                    post_date = post_date_aware.astimezone(timezone(timedelta(hours=8))).replace(tzinfo=None)
                except (ValueError, TypeError):
                    print(f"跳过：无法解析日期: {created_at_str}")
                    continue

                is_pinned = post.get('isTop') == 1
                
                if not (start_date <= post_date <= end_date):
                    if is_pinned:
                        print(f"跳过：置顶微博发布于 {post_date.strftime('%Y-%m-%d')}，不在指定区间内。")
                    else:
                        if post_date < start_date:
                            stop_scraping = True
                            print("已遇到早于起始日期的常规微博，停止翻页。")
                    continue
                
                is_retweet = 'retweeted_status' in post

                if originals_only and is_retweet:
                    print(f"筛选模式: 跳过一条转发微博。")
                    continue
                
                print(f"处理：{'转发' if is_retweet else '原创'}微博发布于 {post_date.strftime('%Y-%m-%d')}，在指定区间内。")
                
                post_id = post.get('id')
                post_comments_count = post.get('comments_count', 0)
                
                all_posts_data.append({
                    'post_id': post_id,
                    'content': post.get('text_raw', ''),
                    'reposts_count': post.get('reposts_count', 0),
                    'comments_count': post_comments_count,
                    'likes_count': post.get('attitudes_count', 0),
                    'created_at': created_at_str,
                    'type': 'retweet' if is_retweet else 'original'
                })
                
                # (评论获取代码已移除)
            
            if stop_scraping:
                break
            
            page += 1
            time.sleep(2)
        except Exception as e:
            print(f"获取微博列表时出错: {e}")
            break

    if not all_posts_data:
        print("在指定日期区间内未找到任何符合条件的微博。")
        return
        
    df_posts = pd.DataFrame(all_posts_data)

    # --- !! 修改点: 定义保存路径 !! ---
    # 注意: 路径中的反斜杠 \ 需要写成 \\
    output_path_posts = "C:\\tongji\\0 code\\00_data\\raw_weibo_posts"
    
    # --- !! 修改点: 自动创建目录 !! ---
    os.makedirs(output_path_posts, exist_ok=True)
    
    # --- !! 修改点: 文件名中加入UID !! ---
    filename_posts = f"weibo_posts_UID_{uid}_{start_date_str}_to_{end_date.strftime('%Y-%m-%d')}.csv"

    # 组装完整路径
    full_path_posts = os.path.join(output_path_posts, filename_posts)

    df_posts.to_csv(full_path_posts, index=False, encoding='utf-8-sig')
    
    print(f"数据已保存到 {full_path_posts}")


# --- 3. 主程序入口 ---
def main():
    """
    主执行函数
    """
    
    # --- 在这里配置你要爬取的博主列表和对应的时间范围 ---
    # 这是一个任务列表，每个字典代表一个独立的爬取任务
    # 你可以复制粘贴字典块来添加更多任务
    
    scrape_tasks = [
        {
            "uid": "7801655101",           # 示例博主1: (UID)
            "start_date": "2024-12-05",    # (开始日期)
            "end_date": "2025-09-30",      # (结束日期)
            "originals_only": False,       # (是否只看原创: True / False)
        },
        {
            "uid": "7915828567",           # 示例博主2: (UID)
            "start_date": "2024-12-05",    # (开始日期)
            "end_date": "2025-09-30",      # (结束日期)
            "originals_only": False,        # (是否只看原创: True / False)
        },
        
        # --- 你可以在下面添加更多任务 ---
        {
            "uid": "7915670982",           # 示例博主3
            "start_date": "2024-12-05",    
            "end_date": "2025-09-30",      
            "originals_only": False,       
        },
    ]
    
    # ----------------- 配置结束 -----------------

    print(f"总共找到 {len(scrape_tasks)} 个爬取任务。")
    
    task_count = len(scrape_tasks)
    for i, task in enumerate(scrape_tasks):
        print(f"\n--- [任务 {i+1}/{task_count}] ---")
        print(f"开始爬取 UID: {task['uid']}，日期: {task['start_date']} 到 {task['end_date']}")
        
        try:
            scrape_official_account(
                uid=task['uid'],
                start_date_str=task['start_date'],
                end_date_str=task['end_date'],
                originals_only=task.get('originals_only', False) # 使用 .get() 增加灵活性
            )
            print(f"--- [任务 {i+1}/{task_count}] UID: {task['uid']} 已完成 ---")
        
        except KeyboardInterrupt:
            print("检测到手动中断(Ctrl+C)。正在停止所有任务...")
            break
        except Exception as e:
            print(f"!!! 爬取 UID: {task['uid']} 时发生严重错误: {e}")
            print("将跳过此任务，继续下一个。")
        
        # 在每个博主任务之间暂停一下，防止被封
        if i < task_count - 1: # 如果不是最后一个任务
            pause_time = 30
            print(f"\n... 暂停 {pause_time} 秒，防止请求过于频繁 ...")
            try:
                time.sleep(pause_time)
            except KeyboardInterrupt:
                print("检测到手动中断(Ctrl+C)。正在停止所有任务...")
                break

    print("\n=====================")
    print("所有爬取任务已执行完毕。")

if __name__ == "__main__":
    main()